# Data Analysis

Now that we've got clean data, let's start with some basic financial analysis.

First, let's load our CSV file into a DataFrame, covert our dates, set the index, and check for duplicated rows or missing values.

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/ImperialCollegeLondon/efds-ta-python/refs/heads/main/pyfi-data/AAPL_2026_clean.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date").sort_index().drop_duplicates()

print("Duplicates:", df.duplicated().sum())
print("Missing:", df.isnull().sum().sum())
print("Columns:", df.columns)

df

## Returns

Returns refer to the gain or loss made on an initial investment. We can use the generic **relative change** formula here:

$$ relative\ change = (value_{new} - value_{original}) / value_{original} $$

### Simple Daily Returns

We can apply this to close prices to calculate the simple daily return:

$$ simple\ daily\ return = (close\ price_{today} - close\ price_{yesterday}) / close\ price_{yesterday} $$

In [ ]:
jan27_closing = df.loc["2026-01-27", "Close"]
jan26_closing = df.loc["2026-01-26", "Close"]

jan27_return = (jan27_closing - jan26_closing) / jan26_closing
print(f"Return on 27 Jan was {jan27_return:.2%}")

This simple daily return expresses a loss in value of 1.12% from one day to the next. Notice we leave our return in decimal form, but it is common to communicate it as a percentage, so for output, we use `f-strings` and `:.2%` to display it as a percentage.

If we wanted to use the above approach to calculate daily returns for each day in our data set, it would take a long time. Let's see how we can use pandas `pct_change()` to make this sort of work easy, by applying our relative change formula one column at a time.

**NOTE** despite its name `pct_change()` does not produce percentage values.

In [ ]:
# Create a new column and populate it with daily returns
df["DailyReturn"] = df["Close"].pct_change()
df

Notice how the first row in our data has a missing value **NaN** in the new daily return column.

<details>
    <summary>Why?</summary>
    
*Because our data doesn't have a close price for the day before it!*
</details>

What to do with this missing value depends on what further analysis we want to do. If we want to carry out simple descriptive statistics like compute the mean, max, or standard deviation,  we can leave our missing value as NaN, because pandas will by default ignore NaNs when calculating these. For other analyses though, we may want to drop or fill this value.

### Aggregating Returns

Aggregating simple daily returns can provide some useful insights:
- The mean daily return gives us a typical day's performance
- The maximum daily return tells us our top performing day
- The standard deviation of daily returns tells us how volatile the period has been

In [ ]:
print(f"Average daily return: {df['DailyReturn'].mean():.2%}")

print(f"Highest daily return: {df['DailyReturn'].max():.2%} on {df['DailyReturn'].idxmax():%d %B %Y}")

print(f"Daily Volatility: {df['DailyReturn'].std():.2%}")

### Cumulative Returns

Now let's go beyond the daily return and look at performance over longer periods. Instead of comparing the price on a given day with the price the day before, we'll look at the price on one date (the date we sell, for example) as it compares to some earlier date (the date we bought, to continue the example). This way we can get an idea of how we are performing over weeks, months or years.

We could apply the daily return formula here again:

$$ period\ return = (close\ price_{sell\ date} - close\ price_{buy\ date}) / close\ price_{buy\ date} $$

But since we have simple daily returns calculated, we can work with these instead. Simple (*arithmetic*) returns compound, which means that if we take all the simple daily returns from our buy date to our sell date, we can get the overall return for that period. Mathematically, we need to first convert our simple returns to growth factors by adding 1 (`+1`), then find the product of these growth factors, then subtract 1 (`-1`) from the result to convert from growth factors back to simple returns.

For a first example, let's assume we buy at the start date in our data and sell on the last day represented in our data. To calculate this period return, we'll need to find the product of *all* the growth factors in our data (and then remember to convert back to simple returns).

In [ ]:
period_return = (1 + df["DailyReturn"]).prod() - 1

print(f"Cumulative return at end of period: {period_return:.2%}")

So if we had bought at the earliest opportunity in the data (the first day's close, since we are using close prices) and sold on the last possible date in our data, we would have seen a gain of **16.98%**.

What if we wanted to see how our initial investment is performing at many different points in the period? For this, we will calculate cumulative returns, into a new column in the dataframe.

In [ ]:
df["CumulativeReturn"] = (1 + df["DailyReturn"]).cumprod() - 1

display(df)

### Exercise: Buy & Hold

#### Part 1

Imagine you had bought AAPL at the start of 2026. What would your expected return have been had you sold at the end of March? At the end of April?

In [ ]:
## YOUR CODE GOES HERE

#### Part 2

What if you had bought on 1 April instead of at the beginning of the year, still selling at the end of the period.

**NOTE** Because we are dealing with *close* prices, we cannot include the return on the date we buy - 2 April is the first return to use.

In [ ]:
## YOUR CODE GOES HERE

## Moving Averages

Moving averages are a different kind of indicator, one that smooths out small variations in trading data to give a better picture of the overall trend.

A Simple Moving Average (SMA) is one which averages out a price over a specific period. The average is "moving" because when a new day is considered in the period, the oldest date is discarded.

Moving averages can be *fast*, when they cover a short period, or *slow* when they consider a longer period. The longer the period, the more those small variations are smoothed out.

In [ ]:
# Calculate a fast, 20-Day Moving Average
df['FastMA'] = df['Close'].rolling(window=20).mean()

# Calculate a slow, 200-Day Moving Average
df['SlowMA'] = df['Close'].rolling(window=200).mean()

## Historical Volatility

Volatility looks at the degree of variance in a stock, and can be helpful for determining risk. We saw that we could calculate daily volatility by taking the standard deviation of our simple daily returns. Instead of a single value, we can calculate a rolling volatility to show how volatility rises and falls over time.

A commonly seen metric for this is `HV20` - the 20-Day rolling standard deviation. Typically we would annualise this value, which we'll look at in a future notebook.

In [ ]:
# Calculate 20-Day Standard Deviation
df['HV20'] = df['DailyReturn'].rolling(window=20).std()
df

## Surges

Surges in price or trading volume can be helpful indicators for traders. One way to define a surge is as an increase on the day before by an amount higher than some defined threshold. That threshold is often defined as some number of standard deviations above the mean. Let's look at price surges of two standard deviations above the mean.

In [ ]:
# Find the mean return
mean_return = df["DailyReturn"].mean()

# Define a threshold as two standard deviations above the mean
return_threshold = mean_return + (df["DailyReturn"].std() * 2)

# Define a condition
condition = df["DailyReturn"] > return_threshold

# Subset the dataframe where daily returns are higher than the threshold
df[condition]

Here we see the first example of what we call *lookahead bias*. If we were trying to spot surges in real-time, we might identify different dates!

The reason for this is that we're using the full data set to produce the aggregate mean and standard deviation. Consider the surge identified on 2 February - we've built a threshold using data until the end of May. We're setting a threshold using data that wouldn't yet be available to us.

To counteract this, we want to perform out aggregations using only the data that is available up to the date we are working with. The `expanding()` function is similar to `rolling()`, instead using all the available data points up until the in-use point in time.

**NOTE** early in the data surges `expanding()` is based on very few data points. Specifying `expanding(min_periods=10)` leaves the threshold undefined until there is sufficient data to trust it.

In [ ]:
# Find the expanding mean return
mean_return = df["DailyReturn"].expanding().mean()

# Define a threshold as two expanding standard deviations above the mean
return_threshold = mean_return + (df["DailyReturn"].expanding().std() * 2)

# Define a condition
condition = df["DailyReturn"] > return_threshold

# Subset the dataframe where daily returns are higher than the threshold
df[condition]